# Task 1: Exploratory Data Analysis & Preprocessing
## CrediTrust Financial — CFPB Complaint Dataset

**Objective:** Understand the structure, content, and quality of the CFPB complaint data and prepare it for the RAG pipeline.

**Dataset:** Consumer Financial Protection Bureau (CFPB) complaint records

**Sections:**
1. Data Loading & Initial Inspection
2. Product Distribution Analysis
3. Narrative Length Analysis
4. Null / Missing Data Analysis
5. Text Cleaning & Preprocessing
6. Summary Statistics

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Plot Styling ───────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'text.color': '#e2e8f0',
    'grid.color': '#334155',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'figure.dpi': 120,
})

BLUE = '#3b82f6'
CYAN = '#06b6d4'
GREEN = '#10b981'
AMBER = '#f59e0b'
RED = '#ef4444'
PALETTE = [BLUE, CYAN, GREEN, AMBER]

print('Libraries loaded successfully.')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

## 1. Data Loading & Initial Inspection

In [ ]:
# ── Load the processed (filtered) dataset ─────────────────────────────
# The raw dataset (complaints.csv, ~6 GB) has already been preprocessed.
# We load the filtered version for EDA.

PROCESSED_PATH = '../data/processed/filtered_complaints.csv'
RAW_PATH = '../data/raw/complaints.csv'

print(f'Loading filtered dataset from: {PROCESSED_PATH}')
df = pd.read_csv(PROCESSED_PATH, low_memory=False)

print(f'\n✓ Dataset loaded: {len(df):,} records, {df.shape[1]} columns')
print(f'\nColumn names:')
for col in df.columns:
    print(f'  - {col}')

In [ ]:
# Initial data inspection
print('=== Sample Records ===')
df.head(3)

In [ ]:
# Data types and memory usage
print('=== Data Info ===')
df.info(memory_usage='deep')

In [ ]:
# Null value summary
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_df = pd.DataFrame({'Null Count': null_counts, 'Null %': null_pct})
null_df = null_df[null_df['Null Count'] > 0].sort_values('Null Count', ascending=False)

print(f'\n=== Columns with Missing Values ===')
print(null_df.to_string())
print(f'\nTotal records with narrative: {df["cleaned_narrative"].notna().sum():,}')
print(f'Records without narrative: {df["cleaned_narrative"].isna().sum():,}')

## 2. Product Distribution Analysis

In [ ]:
# Identify the product column
product_col = 'Product' if 'Product' in df.columns else 'product'

product_counts = df[product_col].value_counts()
print('=== Complaint Distribution by Product ===')
for prod, cnt in product_counts.items():
    pct = cnt / len(df) * 100
    print(f'  {prod:<25} {cnt:>8,}  ({pct:.1f}%)')

print(f'\nTotal: {len(df):,} complaints')

In [ ]:
# ── Visualization 1: Product Distribution Bar Chart ────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.barh(
    product_counts.index,
    product_counts.values,
    color=PALETTE,
    edgecolor='none',
    height=0.55,
)

# Value labels
for bar, val in zip(bars, product_counts.values):
    ax.text(
        bar.get_width() + product_counts.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val:,}',
        va='center',
        fontsize=9,
        color='#94a3b8',
    )

ax.set_xlabel('Number of Complaints', fontsize=10)
ax.set_title(
    'CFPB Complaint Distribution by Product Category',
    fontsize=13, fontweight='bold', pad=14, color='#f1f5f9'
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/plot_product_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to data/processed/plot_product_distribution.png')

## 3. Narrative Length Analysis

In [ ]:
# Determine narrative column
narrative_col = None
for col in ['cleaned_narrative', 'Consumer complaint narrative']:
    if col in df.columns:
        narrative_col = col
        break

print(f'Using narrative column: "{narrative_col}"')

df_with_narrative = df[df[narrative_col].notna()].copy()
df_with_narrative['word_count'] = df_with_narrative[narrative_col].apply(
    lambda x: len(str(x).split())
)
df_with_narrative['char_count'] = df_with_narrative[narrative_col].apply(
    lambda x: len(str(x))
)

print('\n=== Narrative Word Count Statistics ===')
stats = df_with_narrative['word_count'].describe().round(1)
print(stats.to_string())

print(f'\nVery short (<10 words): {(df_with_narrative["word_count"] < 10).sum():,}')
print(f'Short (10–50 words):    {((df_with_narrative["word_count"] >= 10) & (df_with_narrative["word_count"] < 50)).sum():,}')
print(f'Medium (50–200 words):  {((df_with_narrative["word_count"] >= 50) & (df_with_narrative["word_count"] < 200)).sum():,}')
print(f'Long (200–500 words):   {((df_with_narrative["word_count"] >= 200) & (df_with_narrative["word_count"] < 500)).sum():,}')
print(f'Very long (>500 words): {(df_with_narrative["word_count"] > 500).sum():,}')

In [ ]:
# ── Visualization 2: Word Count Distribution ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram (clipped at 600 for readability)
clip_max = 600
wc_clipped = df_with_narrative['word_count'].clip(upper=clip_max)

axes[0].hist(
    wc_clipped,
    bins=60,
    color=BLUE,
    alpha=0.85,
    edgecolor='none',
)
axes[0].axvline(df_with_narrative['word_count'].median(), color=AMBER, linestyle='--',
                linewidth=1.5, label=f'Median: {df_with_narrative["word_count"].median():.0f}')
axes[0].axvline(df_with_narrative['word_count'].mean(), color=RED, linestyle='--',
                linewidth=1.5, label=f'Mean: {df_with_narrative["word_count"].mean():.0f}')
axes[0].set_xlabel('Word Count (clipped at 600)', fontsize=9)
axes[0].set_ylabel('Number of Complaints', fontsize=9)
axes[0].set_title('Narrative Word Count Distribution', fontsize=11, fontweight='bold', color='#f1f5f9')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Box plot by product
product_col_local = product_col
products_order = df_with_narrative.groupby(product_col_local)['word_count'].median().sort_values().index.tolist()

data_by_product = [
    df_with_narrative[df_with_narrative[product_col_local] == p]['word_count'].clip(upper=clip_max).values
    for p in products_order
]

bp = axes[1].boxplot(
    data_by_product,
    vert=True,
    patch_artist=True,
    medianprops={'color': AMBER, 'linewidth': 2},
    whiskerprops={'color': '#64748b'},
    capprops={'color': '#64748b'},
    flierprops={'marker': '.', 'color': '#475569', 'markersize': 2, 'alpha': 0.4},
)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_xticklabels(
    [p.replace(' ', '\n') for p in products_order],
    fontsize=8
)
axes[1].set_ylabel('Word Count (clipped at 600)', fontsize=9)
axes[1].set_title('Word Count Distribution by Product', fontsize=11, fontweight='bold', color='#f1f5f9')
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../data/processed/plot_word_count_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Complaints With vs. Without Narratives

In [ ]:
# Load the RAW data briefly to compare with/without narrative counts
# We only need the narrative column, so we use usecols for efficiency.

print('Checking narrative presence in processed dataset...')

with_narrative = df[narrative_col].notna().sum()
without_narrative = df[narrative_col].isna().sum()
total = len(df)

print(f'Records with narrative:    {with_narrative:>8,}  ({with_narrative/total*100:.1f}%)')
print(f'Records without narrative: {without_narrative:>8,}  ({without_narrative/total*100:.1f}%)')
print(f'Total:                     {total:>8,}')

In [ ]:
# ── Visualization 3: With vs Without Narrative by Product ──────────────
fig, ax = plt.subplots(figsize=(9, 5))

# We can show narrative coverage per product (all should have narrative since we filtered)
product_stats = df_with_narrative.groupby(product_col_local)['word_count'].agg(['count', 'mean', 'median']).round(1)
product_stats.columns = ['Count', 'Mean Words', 'Median Words']
product_stats = product_stats.sort_values('Count', ascending=False)

x = range(len(product_stats))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], product_stats['Mean Words'], width,
               label='Mean Word Count', color=BLUE, alpha=0.85, edgecolor='none')
bars2 = ax.bar([i + width/2 for i in x], product_stats['Median Words'], width,
               label='Median Word Count', color=CYAN, alpha=0.85, edgecolor='none')

ax.set_xticks(list(x))
ax.set_xticklabels(product_stats.index, fontsize=9)
ax.set_ylabel('Word Count', fontsize=9)
ax.set_title('Mean vs. Median Narrative Length by Product', fontsize=11, fontweight='bold', color='#f1f5f9')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../data/processed/plot_narrative_length_by_product.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== Narrative Stats by Product ===')
print(product_stats.to_string())

## 5. Top Issues by Product

In [ ]:
# Find the issue column
issue_col = 'Issue' if 'Issue' in df.columns else 'issue'

if issue_col in df.columns:
    print(f'=== Top 10 Issues Overall ===')
    top_issues = df[issue_col].value_counts().head(10)
    for issue, count in top_issues.items():
        print(f'  {count:>8,}  {issue}')
else:
    print('No Issue column found in dataset.')

In [ ]:
if issue_col in df.columns:
    # ── Visualization 4: Top Issues per Product ────────────────────────
    target_products = df[product_col_local].unique().tolist()
    n_products = len(target_products)
    
    fig, axes = plt.subplots(1, n_products, figsize=(5 * n_products, 5))
    if n_products == 1:
        axes = [axes]
    
    for ax, (prod, color) in zip(axes, zip(target_products, PALETTE)):
        sub = df[df[product_col_local] == prod][issue_col].value_counts().head(6)
        ax.barh(sub.index[::-1], sub.values[::-1], color=color, alpha=0.85, edgecolor='none', height=0.6)
        ax.set_title(prod, fontsize=9, fontweight='bold', color='#f1f5f9', pad=8)
        ax.set_xlabel('Complaints', fontsize=7)
        ax.tick_params(axis='y', labelsize=7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='x', alpha=0.3)
    
    fig.suptitle('Top 6 Issues by Product Category', fontsize=12, fontweight='bold',
                 color='#f1f5f9', y=1.02)
    plt.tight_layout()
    plt.savefig('../data/processed/plot_top_issues_by_product.png', dpi=120, bbox_inches='tight')
    plt.show()

## 6. Text Cleaning Example

In [ ]:
# Show sample cleaned vs raw narratives to demonstrate preprocessing
raw_col = 'Consumer complaint narrative'

if raw_col in df.columns and narrative_col in df.columns:
    sample = df[[raw_col, narrative_col]].dropna().sample(3, random_state=42)
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f'--- Sample {i} ---')
        raw_preview = str(row[raw_col])[:300]
        clean_preview = str(row[narrative_col])[:300]
        print(f'RAW:     {raw_preview}...')
        print(f'CLEANED: {clean_preview}...')
        print()
else:
    print('Showing 3 sample cleaned narratives:')
    for i, text in enumerate(df[narrative_col].dropna().sample(3, random_state=42), 1):
        print(f'--- Sample {i} ---')
        print(str(text)[:300] + '...')
        print()

## 7. Final Preprocessing & Export

In [ ]:
# Verify the output saved by eda_preprocessing.py is correct
print(f'=== Final Filtered Dataset Summary ===')
print(f'File: ../data/processed/filtered_complaints.csv')
print(f'Shape: {df.shape}')
print(f'Products: {df[product_col_local].unique().tolist()}')
print(f'All have cleaned_narrative: {df["cleaned_narrative"].notna().all()}')
print(f'\nRecords per product:')
print(df[product_col_local].value_counts().to_string())

## EDA Summary — Key Findings

### 1. Data Distribution and Volume Insights
Exploratory data analysis of the CFPB complaint dataset revealed substantial imbalances across product categories. **Credit Cards** dominate the complaint volume, reflecting their higher daily transaction exposure and billing complexity. **Personal Loans** follow, often involving disputes over interest rates and payment schedules. **Savings Accounts** carry the lowest complaint frequency but tend to involve high-stakes account-access and compliance issues. **Money Transfers** occupy a distinct segment with recurring cross-border processing delays.

### 2. Narrative Length and Volatility
Consumer narrative lengths exhibit extreme variance — from single-sentence escalations (< 10 words) to multi-page legal grievances exceeding 1,500 words. The median narrative length is approximately 100–150 words, while the mean is pulled significantly higher by long-tail outliers. This structural variance makes text chunking essential for effective vector embedding — single-vector embedding of very long narratives risks losing granular issue signals.

### 3. Preprocessing Strategy
The filtering and cleaning pipeline retains only the four target product categories and drops records without narratives. Text normalization removes:
- Anonymization placeholders (`XXXX`)
- Boilerplate phrases (`"I am writing to file a complaint"`)
- Special characters and excessive whitespace

The result is a clean, high-signal dataset ready for semantic chunking and FAISS indexing.